# 01 — Data Cleaning

Cleans raw Play Store exports for HealthifyMe and MyFitnessPal using the
functions in `src/clean_data.py`. This notebook runs the full pipeline,
prints a step-by-step cleaning log, and saves outputs to `data/processed/`.

**Outputs produced**
- `data/processed/healthifyme_cleaned.csv`
- `data/processed/myfitnesspal_cleaned.csv`
- `data/processed/combined_cleaned.csv`

In [ ]:
import sys
import os

# Ensure the repo root is on the path so src/ imports work
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import pandas as pd
from IPython.display import display

from src.clean_data import clean_dataset, build_cleaning_log_df

## 1  HealthifyMe

Run every cleaning step on the raw HealthifyMe export (65,518 rows in).
The pipeline applies steps in this order:

1. **Drop all-null columns** — `review_title` is 100% null in both datasets; dropped immediately.
2. **Fix `developer_response`** — mixed str/float dtype warning suppressed with `low_memory=False`; NaN sentinels (`"nan"`, `"0"`) replaced with `pd.NA`; boolean `has_dev_response` added (38,334 rows have a response, 58.5%).
3. **Drop null review text** — 13 rows with null/empty `review_description` dropped.
4. **Parse `review_date`** — string `YYYY-MM-DD HH:MM:SS` → `datetime64[ns]`; 0 parsing failures.
5. **Flag `possible_mixed_sentiment`** — rating ≤ 3 + temporal/contrast regex (e.g. "used to be", "but now", "was great … but"). **967 rows flagged; none dropped.**
6. **Language detection** — `langid` run only on the 4,672 non-ASCII rows (~7%); all ASCII-only rows labelled `'en'` directly. Non-English detected: **2,358 rows (3.60%)**. Top non-EN: ja 641, zh 343, hr 157, ko 146.
7. **Filter non-English** — 2,358 rows dropped.
8. **Add `app_label`** — value `'healthifyme'`.

**Expected output: 65,518 → 63,147 rows (96.4% retained)**

In [ ]:
hfy_df, hfy_log = clean_dataset(
    path="../data/raw/healthifyme_reviews.csv",
    app_name="healthifyme",
    filter_english=True,
)

### HealthifyMe — Cleaning log

In [ ]:
hfy_log_df = build_cleaning_log_df(hfy_log)
display(hfy_log_df.style.set_properties(**{"text-align": "left"}))

### HealthifyMe — Before / after summary

In [ ]:
rows_in_hfy  = hfy_log_df.loc[hfy_log_df["step"] == "load_raw", "rows_before"].iloc[0]
rows_out_hfy = len(hfy_df)

print(f"Rows in  : {rows_in_hfy:>8,}")
print(f"Rows out : {rows_out_hfy:>8,}")
print(f"Dropped  : {rows_in_hfy - rows_out_hfy:>8,}  ({(rows_in_hfy - rows_out_hfy)/rows_in_hfy*100:.1f}%)")
print(f"Retained : {rows_out_hfy/rows_in_hfy*100:.1f}%")
print(f"\nColumns  : {list(hfy_df.columns)}")
print(f"\nDtypes:")
print(hfy_df.dtypes)

### HealthifyMe — Flag counts

In [ ]:
print("has_dev_response:")
print(hfy_df["has_dev_response"].value_counts().to_string())

print("\npossible_mixed_sentiment:")
print(hfy_df["possible_mixed_sentiment"].value_counts().to_string())

print("\ndetected_lang value counts (top 15):")
print(hfy_df["detected_lang"].value_counts().head(15).to_string())

### HealthifyMe — Sample of mixed-sentiment flagged rows

In [ ]:
with pd.option_context("display.max_colwidth", 120):
    display(
        hfy_df[hfy_df["possible_mixed_sentiment"] == True][["rating", "review_description"]]
        .sample(5, random_state=42)
    )

## 2  MyFitnessPal

Same 8-step pipeline on the MyFitnessPal export (661,512 rows in). Notable differences from HealthifyMe:

- **Null text**: 4,429 rows dropped (vs 13 for HFY) — MFP has more short/empty submissions.
- **`has_dev_response`**: only 70,421 rows (10.6%) have a dev reply, vs 58.5% for HFY.
- **`possible_mixed_sentiment`**: 19,253 flagged (vs 967) — proportionally similar but MFP has many more "used to be great, now paywalled" complaints.
- **Language detection**: 12,028 non-ASCII rows scanned; **3,433 non-English (0.52%)**. Top non-EN: ja 604, zh 355, es 269, ar 207.

**Expected output: 661,512 → 653,650 rows (98.8% retained)**

In [ ]:
mfp_df, mfp_log = clean_dataset(
    path="../data/raw/myfitnesspal_reviews.csv",
    app_name="myfitnesspal",
    filter_english=True,
)

### MyFitnessPal — Cleaning log

In [ ]:
mfp_log_df = build_cleaning_log_df(mfp_log)
display(mfp_log_df.style.set_properties(**{"text-align": "left"}))

### MyFitnessPal — Before / after summary

In [ ]:
rows_in_mfp  = mfp_log_df.loc[mfp_log_df["step"] == "load_raw", "rows_before"].iloc[0]
rows_out_mfp = len(mfp_df)

print(f"Rows in  : {rows_in_mfp:>8,}")
print(f"Rows out : {rows_out_mfp:>8,}")
print(f"Dropped  : {rows_in_mfp - rows_out_mfp:>8,}  ({(rows_in_mfp - rows_out_mfp)/rows_in_mfp*100:.1f}%)")
print(f"Retained : {rows_out_mfp/rows_in_mfp*100:.1f}%")
print(f"\nColumns  : {list(mfp_df.columns)}")

### MyFitnessPal — Flag counts

In [ ]:
print("has_dev_response:")
print(mfp_df["has_dev_response"].value_counts().to_string())

print("\npossible_mixed_sentiment:")
print(mfp_df["possible_mixed_sentiment"].value_counts().to_string())

print("\ndetected_lang value counts (top 15):")
print(mfp_df["detected_lang"].value_counts().head(15).to_string())

### MyFitnessPal — Sample of mixed-sentiment flagged rows

In [ ]:
with pd.option_context("display.max_colwidth", 120):
    display(
        mfp_df[mfp_df["possible_mixed_sentiment"] == True][["rating", "review_description"]]
        .sample(5, random_state=42)
    )

## 3  Save cleaned files

Three files written to `data/processed/`:
- Per-app files for independent modelling (BERTopic, sentiment)
- Combined file for cross-app comparison tables

Note: `data/processed/` is gitignored — these are derived artefacts, reproducible from the pipeline.

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)

hfy_out = "../data/processed/healthifyme_cleaned.csv"
mfp_out = "../data/processed/myfitnesspal_cleaned.csv"
combined_out = "../data/processed/combined_cleaned.csv"

hfy_df.to_csv(hfy_out, index=False)
mfp_df.to_csv(mfp_out, index=False)

combined = pd.concat([hfy_df, mfp_df], ignore_index=True)
combined.to_csv(combined_out, index=False)

print(f"Saved: {hfy_out}  ({len(hfy_df):,} rows)")
print(f"Saved: {mfp_out}  ({len(mfp_df):,} rows)")
print(f"Saved: {combined_out}  ({len(combined):,} rows)")

## 4  Combined dataset overview

In [ ]:
print("Combined shape:", combined.shape)
print("\nRows per app:")
print(combined["app_label"].value_counts().to_string())

print("\nRating distribution per app:")
display(
    combined.groupby(["app_label", "rating"])
    .size()
    .rename("count")
    .reset_index()
    .pivot(index="rating", columns="app_label", values="count")
    .fillna(0)
    .astype(int)
)

print("\nDate range per app:")
for app, grp in combined.groupby("app_label"):
    print(f"  {app}: {grp['review_date'].min()}  →  {grp['review_date'].max()}")

print("\nMixed-sentiment flags per app:")
print(
    combined.groupby("app_label")["possible_mixed_sentiment"]
    .sum()
    .astype(int)
    .to_string()
)

## 5  Full cleaning summary — both apps side by side

Quick reference for what was dropped at each step and why.

| Step | HFY dropped | MFP dropped | Reason |
|---|---|---|---|
| drop_useless_columns | 0 rows | 0 rows | `review_title` column removed (100% null) |
| fix_developer_response | 0 rows | 0 rows | dtype fix + `has_dev_response` flag added |
| drop_null_review_text | 13 | 4,429 | No text = nothing to analyse |
| parse_review_date | 0 | 0 | All dates parsed cleanly |
| add_mixed_sentiment_flag | 0 | 0 | Flag only — 967 / 19,253 tagged |
| run_language_detection | 0 | 0 | Detection pass; counts reported |
| filter_non_english | 2,358 | 3,433 | Non-Latin-script reviews (3.6% / 0.52%) |
| **Total** | **2,371** | **7,862** | |

In [ ]:
hfy_log_df["app"] = "healthifyme"
mfp_log_df["app"] = "myfitnesspal"

full_log = pd.concat([hfy_log_df, mfp_log_df], ignore_index=True)
cols = ["app", "step", "rows_before", "rows_after", "rows_dropped", "detail"]
with pd.option_context("display.max_colwidth", 100, "display.max_rows", 40):
    display(full_log[cols])